# 🏈 Fantasy Football Analytics: Floor/Ceiling, Value Over Replacement & Game Script

This notebook showcases the **data-science engine** behind a personal fantasy-football
monitoring tool. The live tool watches my ESPN leagues, diffs rosters day-over-day, and
pushes Discord alerts on a GitHub Actions schedule — but that part needs private league
cookies and a running service, so it lives in the repo, not here.

**What runs in this notebook is the interesting, reproducible part** — the models, built
entirely on *public* NFL data (`nfl_data_py`) and ESPN's *public* scoreboard API. No
accounts, no secrets, no private data.

Three ideas, each turned from a hand-picked guess into something data-driven:

1. **Monte Carlo floor/ceiling** — turn a single projection into a distribution (is 12
   points a safe 10–14 or a boom/bust 3–28?).
2. **Value over replacement (VORP)** — make points comparable *across positions* so a
   9-point TE can correctly outrank a 9-point WR.
3. **"Heating up" thresholds** — learn, per position, how big a recent-form jump is
   actually *notable* instead of guessing.

Plus a live bonus: **game-script flags** from real Vegas point spreads.

> Full source (the live monitor, GitHub Actions workflow, alerting): **[github.com/jamesonjdavis-maker/fantasy-football-monitor](https://github.com/jamesonjdavis-maker/fantasy-football-monitor)**

*Note: this notebook is a showcase / explainer of the methodology. The private, scheduled
monitoring service stays in the repo where it belongs.*

## Setup

On Kaggle, turn **Internet** on (Settings → Internet) so we can fetch the public NFL data.

We read nflverse's public data files directly with `pandas` (Kaggle already has `pandas`,
`numpy`, `matplotlib`, and `pyarrow`), so there's **nothing to pip-install** and no fragile
build step.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
pd.set_option('display.width', 120)
print('imports ok')

## Load public NFL history

nflverse publishes weekly player stats as public Parquet files on GitHub Releases. We read
the last several **completed** seasons directly, season-by-season, skipping any that aren't
published yet (a future or too-early season 404s). Same resilient approach the live tool
uses — one missing season never breaks the pull.

In [ ]:
POSITIONS = ('QB', 'RB', 'WR', 'TE')
NFLVERSE = ('https://github.com/nflverse/nflverse-data/releases/'
            'download/player_stats/player_stats_{yr}.parquet')

def completed_seasons(n=6):
    from datetime import date
    today = date.today()
    current = today.year if today.month >= 8 else today.year - 1
    return list(range(current - n, current))

def load_weekly(seasons):
    frames, used = [], []
    for yr in seasons:
        try:
            frames.append(pd.read_parquet(NFLVERSE.format(yr=yr)))
            used.append(yr)
        except Exception as e:
            print(f'  {yr} unavailable ({type(e).__name__}); skipping')
    if not frames:
        raise RuntimeError('no seasons could be loaded')
    print('loaded seasons:', used)
    return pd.concat(frames, ignore_index=True)

SEASONS = completed_seasons(6)
raw = load_weekly(SEASONS)

df = raw.copy()
if 'season_type' in df.columns:
    df = df[df['season_type'] == 'REG']
df = df[df['position'].isin(POSITIONS)].copy()
df['ppr'] = pd.to_numeric(df.get('fantasy_points_ppr', 0), errors='coerce').fillna(0.0)
print(f'{len(df):,} player-weeks across {df["season"].nunique()} seasons')
df[['player_display_name', 'position', 'recent_team', 'season', 'week', 'ppr']].head()

## 1. Monte Carlo floor / ceiling

A single projected number hides risk. The idea:

1. From history, learn each position's **performance-multiplier** distribution —
   `weekly points ÷ that player's season average`. This is a dimensionless boom/bust
   *shape* (WRs swing harder than RBs) we can transfer onto any projection.
2. For a player projected `μ` points, simulate `N` outcomes = `μ × sampled multipliers`,
   and read the **10th / 50th / 90th** percentiles as **floor / median / ceiling**.

We also tier by scoring level (low/mid/high): lower-scored players are relatively boomier
(more zeros, occasional spikes); studs are steadier.

In [ ]:
MIN_SEASON_AVG = 6.0  # only shape variance from real contributors

def tier(avg):
    if avg is None: return 'mid'
    if avg < 9:  return 'low'
    if avg < 14: return 'mid'
    return 'high'

m = df.copy()
m['season_avg'] = m.groupby(['player_id', 'season'])['ppr'].transform('mean')
m = m[m['season_avg'] >= MIN_SEASON_AVG]
m['mult'] = m['ppr'] / m['season_avg']
m['tier'] = m['season_avg'].apply(tier)

# (position, tier) -> array of multipliers, with a pooled (pos, 'all') fallback
mult_dist = {}
for pos in POSITIONS:
    pos_df = m[m['position'] == pos]
    mult_dist[(pos, 'all')] = pos_df['mult'].to_numpy()
    for t in ('low', 'mid', 'high'):
        arr = pos_df.loc[pos_df['tier'] == t, 'mult'].to_numpy()
        if len(arr) >= 200:
            mult_dist[(pos, t)] = arr[np.isfinite(arr)]

print('multiplier pools built:', len(mult_dist))

In [ ]:
N_SIMS = 5000

def simulate(projection, position, seed=0):
    if projection is None or projection <= 0:
        return None
    pos = (position or '').upper()
    mults = mult_dist.get((pos, tier(projection)))
    if mults is None or len(mults) == 0:
        mults = mult_dist.get((pos, 'all'))
    if mults is None or len(mults) == 0:
        return None
    rng = np.random.default_rng(seed)
    sims = projection * rng.choice(mults, size=N_SIMS)
    floor, median, ceiling = np.percentile(sims, [10, 50, 90])
    spread = (ceiling - floor) / (median or 0.01)
    if spread >= 1.6:            label = 'boom/bust'
    elif floor / (median or 0.01) >= 0.65: label = 'safe floor'
    else:                        label = 'balanced'
    return dict(proj=round(projection, 1), floor=round(floor, 1),
                median=round(median, 1), ceiling=round(ceiling, 1),
                label=label, sims=sims)

# Same projection (12 pts), different positions -> different risk shapes
for pos in POSITIONS:
    r = simulate(12.0, pos)
    print(f'{pos}: floor {r["floor"]:5.1f} | median {r["median"]:5.1f} | '
          f'ceiling {r["ceiling"]:5.1f}  -> {r["label"]}')

In [ ]:
# Visualize the distributions behind a 12-point projection
fig, ax = plt.subplots()
for pos in POSITIONS:
    r = simulate(12.0, pos)
    ax.hist(r['sims'], bins=40, histtype='step', linewidth=2, label=pos, density=True)
ax.axvline(12.0, color='k', ls='--', alpha=0.6, label='projection (12)')
ax.set_title('Simulated weekly outcomes for a 12-point projection, by position')
ax.set_xlabel('fantasy points (PPR)'); ax.set_ylabel('density'); ax.legend()
ax.set_xlim(0, 40)
plt.show()

In [ ]:
# Floor / median / ceiling as a range chart for a few example projections
examples = [('QB', 18), ('RB', 12), ('WR', 12), ('WR', 7), ('TE', 8)]
rows = [simulate(mu, pos) | {'name': f'{pos} ({mu} proj)'} for pos, mu in examples]
fig, ax = plt.subplots()
y = range(len(rows))
for i, r in enumerate(rows):
    ax.plot([r['floor'], r['ceiling']], [i, i], color='#888', lw=6, solid_capstyle='round')
    ax.plot(r['median'], i, 'o', color='#1971c2', ms=9, zorder=3)
    ax.plot(r['proj'], i, '|', color='k', ms=18, mew=2)
    ax.text(r['ceiling'] + 0.4, i, r['label'], va='center', fontsize=9, color='#555')
ax.set_yticks(list(y)); ax.set_yticklabels([r['name'] for r in rows])
ax.set_xlabel('fantasy points  (bar = floor→ceiling, dot = median, tick = projection)')
ax.set_title('Floor / ceiling ranges from Monte Carlo simulation')
plt.tight_layout(); plt.show()

## 2. Value over replacement (VORP)

12 raw points at QB is not the same as 12 at TE — replacement level differs by position.
For a league's starting requirements we find the **last startable rank** at each position
(`teams × starters`, splitting FLEX demand across RB/WR/TE), then measure the weekly PPG a
player *at that rank* historically scores. Subtracting it makes points comparable across
positions, so waiver value ranks fairly.

In [ ]:
def ranks_from_settings(team_count, starters):
    flex  = starters.get('FLEX', 0)
    sflex = starters.get('SUPERFLEX', 0)
    r = lambda base, extra: max(1, round(team_count * (base + extra)))
    return {'QB': r(starters.get('QB', 1), sflex * 0.5),
            'RB': r(starters.get('RB', 2), flex * 0.4),
            'WR': r(starters.get('WR', 2), flex * 0.4),
            'TE': r(starters.get('TE', 1), flex * 0.2)}

def replacement_levels(ranks):
    levels = {}
    for pos, rank in ranks.items():
        pos_df = df[df['position'] == pos]
        def at_rank(s):  # s = that week's PPR values for the position
            ranked = np.sort(s.to_numpy())[::-1]
            return float(ranked[min(rank - 1, len(ranked) - 1)]) if len(ranked) else 0.0
        per_week = pos_df.groupby(['season', 'week'])['ppr'].apply(at_rank)
        levels[pos] = round(float(per_week.mean()), 1) if len(per_week) else 0.0
    return levels

# Standard 12-team (1QB/2RB/2WR/1TE/1FLEX)
std_ranks = ranks_from_settings(12, {'QB':1,'RB':2,'WR':2,'TE':1,'FLEX':1})
std_levels = replacement_levels(std_ranks)
print('12-team ranks       :', std_ranks)
print('replacement PPG     :', std_levels)

In [ ]:
# Superflex raises QB demand -> lowers the QB replacement bar (streamers matter more)
sf_ranks  = ranks_from_settings(12, {'QB':1,'RB':2,'WR':2,'TE':1,'FLEX':1,'SUPERFLEX':1})
sf_levels = replacement_levels(sf_ranks)

labels = list(POSITIONS)
x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots()
ax.bar(x - w/2, [std_levels[p] for p in labels], w, label='standard (1 QB)')
ax.bar(x + w/2, [sf_levels[p]  for p in labels], w, label='superflex (2 QB)')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('replacement-level weekly PPG')
ax.set_title('Replacement level by position and league format')
ax.legend()
for i, p in enumerate(labels):
    ax.text(i - w/2, std_levels[p], f'{std_levels[p]:.1f}', ha='center', va='bottom', fontsize=8)
    ax.text(i + w/2, sf_levels[p],  f'{sf_levels[p]:.1f}',  ha='center', va='bottom', fontsize=8)
plt.show()
print('Note how QB replacement rises in superflex — a startable QB is scarcer.')

In [ ]:
# VORP in action: rank the same raw projection across positions
def vorp(proj, pos, levels):
    return round(proj - levels.get(pos.upper(), 0.0), 1)

candidates = [('QB', 12), ('RB', 12), ('WR', 12), ('TE', 12), ('TE', 9)]
tbl = pd.DataFrame([{'player': f'{p} proj {mu}', 'position': p, 'proj': mu,
                     'replacement': std_levels[p], 'value_over_repl': vorp(mu, p, std_levels)}
                    for p, mu in candidates])
tbl.sort_values('value_over_repl', ascending=False).reset_index(drop=True)

## 3. "Heating up" thresholds

How big a recent-form jump is actually *notable*? Rather than guess, we learn it. For each
player-season we compute a **3-week rolling PPG** vs their **season-to-date average**; the
positive gap (`delta`) is a player producing above their own baseline lately — exactly what
we can also measure live from ESPN. The per-position **75th percentile** of those positive
deltas becomes the bar a current player must clear to get a 🔥 *Heating up* flag.

In [ ]:
MIN_ROLLING_PPG = 4.0
h = df.sort_values(['player_id', 'season', 'week']).copy()
grp = h.groupby(['player_id', 'season'])['ppr']
h['roll3']      = grp.transform(lambda s: s.rolling(3, min_periods=2).mean())
h['season_avg'] = grp.transform(lambda s: s.expanding().mean())
h['delta']      = h['roll3'] - h['season_avg']

notable = h[(h['roll3'] >= MIN_ROLLING_PPG) & (h['delta'] > 0)]
thresholds = notable.groupby('position')['delta'].quantile(0.75).round(2)
thresholds = thresholds.reindex(POSITIONS).fillna(3.0)
print('Notable-rise thresholds (75th pct of positive deltas), PPG:')
print(thresholds.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, pos in zip(axes.ravel(), POSITIONS):
    d = notable.loc[notable['position'] == pos, 'delta']
    ax.hist(d, bins=40, color='#e8590c', alpha=0.75)
    ax.axvline(thresholds[pos], color='k', ls='--', lw=2)
    ax.set_title(f'{pos}  (75th pct = +{thresholds[pos]:.2f} PPG)')
    ax.set_xlim(0, 12)
for ax in axes[-1]:
    ax.set_xlabel('recent 3-wk PPG − season avg')
fig.suptitle('How large a recent-form rise counts as "heating up", by position')
plt.tight_layout(); plt.show()

## 4. Game-script flags from live Vegas spreads

A heavy favorite tends to run more late (RB-friendly); a heavy underdog throws to catch up
(RBs fade, pass-catchers get garbage-time volume). ESPN's **public** scoreboard API gives
point spreads with no key. We pull the current slate and flag lopsided games
(`|spread| ≥ 7`). *(Live cell — output depends on the current NFL week.)*

In [ ]:
SCOREBOARD = 'https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard'
ALIASES = {'WAS': 'WSH', 'JAC': 'JAX', 'LA': 'LAR'}
import re
DETAILS_RE = re.compile(r'^([A-Z]{2,4})\s+(-?\d+(?:\.\d+)?)$')
norm = lambda a: ALIASES.get((a or '').upper(), (a or '').upper()) or None

def fetch_spreads(week=None):
    params = {'week': str(week)} if week else {}
    try:
        data = requests.get(SCOREBOARD, params=params, timeout=30).json()
    except Exception as e:
        print('odds fetch failed:', e); return {}
    spreads = {}
    for event in data.get('events', []):
        for comp in event.get('competitions', []):
            comps = comp.get('competitors', []); odds = comp.get('odds') or []
            if len(comps) < 2 or not odds: continue
            home = next((c for c in comps if c.get('homeAway') == 'home'), None)
            away = next((c for c in comps if c.get('homeAway') == 'away'), None)
            if not home or not away: continue
            ha, aa = norm(home['team'].get('abbreviation')), norm(away['team'].get('abbreviation'))
            det = (odds[0].get('details') or '').strip()
            if det.upper() in ('EVEN', 'PK', 'PICK'):
                spreads[ha] = spreads[aa] = 0.0; continue
            mo = DETAILS_RE.match(det)
            if not mo: continue
            fav, line = norm(mo.group(1)), float(mo.group(2))
            other = aa if fav == ha else ha
            spreads[fav] = line; spreads[other] = -line
    return spreads

spreads = fetch_spreads()  # current slate
print(f'{len(spreads)} teams with spreads')
sorted(spreads.items(), key=lambda kv: kv[1])[:6]  # biggest favorites

In [ ]:
GAME_SCRIPT_SPREAD = 7.0

def game_script_flags(spreads, threshold=GAME_SCRIPT_SPREAD):
    flags = []
    for team, sp in spreads.items():
        if abs(sp) < threshold: continue
        if sp <= -threshold:   # favored
            flags.append((team, sp, 'RB', 'favored — positive script (run-heavy)'))
        else:                  # underdog
            flags.append((team, sp, 'RB', 'underdog — carries may thin'))
            flags.append((team, sp, 'WR/TE', 'underdog — garbage-time target upside'))
    return flags

flags = game_script_flags(spreads)
if flags:
    for team, sp, pos, note in sorted(flags, key=lambda f: f[1]):
        sign = 'fav' if sp < 0 else 'dog'
        print(f'{team:>3} {sign} by {abs(sp):4.1f}  [{pos:5}]  {note}')
else:
    print('No lopsided games on the current slate (or odds not posted yet).')

## Putting it together

In the live tool these combine into a single waiver **value score**:

```
value = (projection − replacement[pos])          # VORP: cross-position fairness
        + 0.3 × (Monte Carlo ceiling − proj)     # reward upside
        + 0.5 × (recent PPG − season avg)         # reward hot form
        + 3.0  if the position is a roster need   # scarcity bonus
```

Everything above is built from **public** data and reproducible here. The pieces that
*can't* live on Kaggle — private-league auth, day-over-day roster diffing, Discord alerts,
the GitHub Actions schedule — are the engineering, and they live in the repo:

### 🔗 [github.com/jamesonjdavis-maker/fantasy-football-monitor](https://github.com/jamesonjdavis-maker/fantasy-football-monitor)

**Honest scope:** this is a heuristics-and-simulation engine calibrated on historical data,
not a trained play-by-play model — the free public feeds don't carry current-season
opportunity data (air yards, snap shares) at the granularity a usage model would need. The
value here is transparent, well-calibrated signals you can actually reason about.